In [1]:
# CELL 1: Install dependencies
!pip install glymur scikit-image tqdm -q
!apt-get install -y libopenjp2-7 libopenjp2-tools -qq
print("✅ Dependencies installed!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 48.2 MB/s eta 0:00:00
Selecting previously unselected package libopenjp2-tools.
(Reading database ... 117540 files and directories currently installed.)
Preparing to unpack .../libopenjp2-tools_2.4.0-6ubuntu0.4_amd64.deb ...
Unpacking libopenjp2-tools (2.4.0-6ubuntu0.4) ...
Setting up libopenjp2-tools (2.4.0-6ubuntu0.4) ...
Processing triggers for man-db (2.10.2-1) ...
✅ Dependencies installed!


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# UPDATE THIS PATH to match your Drive structure
DATA_ROOT = '/content/drive/MyDrive/BreakHis Original'  # <-- Change this!

# Verify the path exists
import os
if os.path.exists(DATA_ROOT):
    print(f"✓ Found dataset at: {DATA_ROOT}")
    # List contents
    print("Contents:", os.listdir(DATA_ROOT))
else:
    print(f"✗ ERROR: Path not found: {DATA_ROOT}")
    print("Check your Google Drive and update DATA_ROOT")

✓ Found dataset at: /content/drive/MyDrive/BreakHis Original
Contents: ['README.txt', 'count_files.sh', 'malignant', 'benign']


In [ ]:
# CELL 3B: Let's see your actual folder structure
import os

DATA_ROOT = '/content/drive/MyDrive/BreakHis/BreakHis_Final_Split2'

def show_structure(path, indent=0):
    """Show folder structure up to 3 levels deep"""
    if indent > 3:
        return
    try:
        items = os.listdir(path)
        for item in items[:10]:  # Limit to first 10 items
            item_path = os.path.join(path, item)
            if os.path.isdir(item_path):
                print("  " * indent + f"📁 {item}/")
                show_structure(item_path, indent + 1)
            else:
                print("  " * indent + f"📄 {item}")
        if len(items) > 10:
            print("  " * indent + f"... and {len(items) - 10} more items")
    except Exception as e:
        print(f"Error: {e}")

print(f"Structure of: {DATA_ROOT}\n")
show_structure(DATA_ROOT)

Structure of: /content/drive/MyDrive/BreakHis Original

📄 README.txt
📄 count_files.sh
📁 malignant/
  📄 ductal_carcinoma.stat.txt.SOB
  📄 process_db_stat.py
  📄 ductal_carcinoma.stat.txt
  📄 lobular_carcinoma.stat.txt
  📄 mucinous_carcinoma.stat.txt
  📄 papillary_carcinoma.stat.txt
  📁 SOB/
    📁 mucinous_carcinoma/
      📁 SOB_M_MC_14-16456/
      📁 SOB_M_MC_14-13418DE/
      📁 SOB_M_MC_14-12773/
      📁 SOB_M_MC_14-13413/
      📁 SOB_M_MC_14-10147/
      📁 SOB_M_MC_14-18842D/
      📁 SOB_M_MC_14-18842/
      📁 SOB_M_MC_14-19979/
      📁 SOB_M_MC_14-19979C/
    📁 ductal_carcinoma/
      📁 SOB_M_DC_14-10926/
      📁 SOB_M_DC_14-11031/
      📁 SOB_M_DC_14-12312/
      📁 SOB_M_DC_14-13412/
      📁 SOB_M_DC_14-11520/
      📁 SOB_M_DC_14-11951/
      📁 SOB_M_DC_14-13993/
      📁 SOB_M_DC_14-15572/
      📁 SOB_M_DC_14-14926/
      📁 SOB_M_DC_14-14946/
      ... and 28 more items
    📁 lobular_carcinoma/
      📁 SOB_M_LC_14-12204/
      📁 SOB_M_LC_14-13412/
      📁 SOB_M_LC_14-15570/
      📁 

In [ ]:
# CELL 4 FIXED: Validation for ORIGINAL BreakHis folder structure
# Structure: class/SOB/subtype/patient_id/magnification/images.png

import numpy as np
import cv2
import glymur
import tempfile
import os
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim

DATA_ROOT = '/content/drive/MyDrive/BreakHis Original'

COMPRESSION_RATIOS = [15, 25, 50, 100, 500, 1000, 5000, 10000, 11000]

# Find a sample image with original BreakHis structure:
# class/SOB/subtype/patient_id/magnification/image.png
sample_img = None

for class_name in ['benign', 'malignant']:
    sob_dir = os.path.join(DATA_ROOT, class_name, 'SOB')
    if not os.path.exists(sob_dir):
        continue

    # Get subtypes (adenosis, ductal_carcinoma, etc.)
    subtypes = [d for d in os.listdir(sob_dir) if os.path.isdir(os.path.join(sob_dir, d))]

    for subtype in subtypes:
        subtype_dir = os.path.join(sob_dir, subtype)

        # Get patient folders (SOB_B_A_14-22549AB, etc.)
        patients = [d for d in os.listdir(subtype_dir) if os.path.isdir(os.path.join(subtype_dir, d))]

        for patient in patients:
            patient_dir = os.path.join(subtype_dir, patient)

            # Get magnification folders (40X, 100X, 200X, 400X)
            mags = [d for d in os.listdir(patient_dir) if os.path.isdir(os.path.join(patient_dir, d))]

            for mag in mags:
                mag_dir = os.path.join(patient_dir, mag)

                # Get image files
                files = [f for f in os.listdir(mag_dir) if f.lower().endswith('.png')]

                if files:
                    sample_img = os.path.join(mag_dir, files[0])
                    print(f"✅ Found image!")
                    print(f"   Path: {class_name}/SOB/{subtype}/{patient}/{mag}/")
                    print(f"   File: {files[0]}")
                    break
            if sample_img:
                break
        if sample_img:
            break
    if sample_img:
        break

if sample_img is None:
    print("❌ No images found!")
    print("\nLet's search recursively...")

    # Fallback: recursive search
    for root, dirs, files in os.walk(DATA_ROOT):
        image_files = [f for f in files if f.lower().endswith('.png')]
        if image_files:
            sample_img = os.path.join(root, image_files[0])
            print(f"✅ Found image at: {root}")
            print(f"   File: {image_files[0]}")
            break

if sample_img is None:
    print("❌ No PNG images found anywhere!")
else:
    print(f"\n{'='*80}")
    print(f"COMPRESSION VALIDATION TEST")
    print(f"{'='*80}\n")

    # Load image
    img = cv2.imread(sample_img)
    if img is None:
        print(f"❌ Could not load image: {sample_img}")
    else:
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w, c = img_rgb.shape
        original_bytes = h * w * c

        print(f"Image size: {w}x{h}x{c} = {original_bytes:,} bytes ({original_bytes/1024:.1f} KB)")
        print(f"This should be ~700x460 for original BreakHis\n")

        print(f"{'CR':<10} {'Achieved':<12} {'Size(bytes)':<12} {'BPP':<10} {'PSNR(dB)':<12} {'SSIM':<10} {'Status'}")
        print("-"*80)

        for cr in COMPRESSION_RATIOS:
            temp_path = tempfile.mktemp(suffix='.jp2')
            try:
                glymur.Jp2k(temp_path, data=img_rgb, cratios=[cr])
                compressed_size = os.path.getsize(temp_path)

                jp2 = glymur.Jp2k(temp_path)
                decompressed = jp2[:]
                if decompressed.dtype != np.uint8:
                    decompressed = np.clip(decompressed, 0, 255).astype(np.uint8)

                achieved_cr = original_bytes / compressed_size
                bpp = (compressed_size * 8) / (h * w)
                psnr_val = psnr(img_rgb, decompressed, data_range=255)
                ssim_val = ssim(img_rgb, decompressed, channel_axis=2, data_range=255)

                if psnr_val < 20:
                    status = "⚠️ UNUSABLE"
                elif psnr_val < 25:
                    status = "⚠️ VERY POOR"
                elif psnr_val < 30:
                    status = "⚠️ POOR"
                elif psnr_val < 40:
                    status = "OK"
                else:
                    status = "✓ GOOD"

                cr_name = f"CR{cr}" if cr < 1000 else f"CR{cr//1000}K"
                print(f"{cr_name:<10} {achieved_cr:<12.1f} {compressed_size:<12} {bpp:<10.4f} {psnr_val:<12.2f} {ssim_val:<10.4f} {status}")

            except Exception as e:
                print(f"CR{cr}: FAILED - {e}")
            finally:
                if os.path.exists(temp_path):
                    os.remove(temp_path)

        print("\n" + "="*80)
        print("REFERENCE: Jo et al. (2021) PSNR for 16-bit mammograms:")
        print("CR15:99dB | CR100:75dB | CR1K:64dB | CR5K:57dB | CR10K:56dB")
        print("="*80)

        # Check if image size is correct
        if w >= 700 and h >= 460:
            print("\n✅ Image size looks correct for original BreakHis!")
        else:
            print(f"\n⚠️ Image size ({w}x{h}) is smaller than expected (700x460)")
            print("   Make sure you're using the original unresized images")

✅ Found image!
   Path: benign/SOB/adenosis/SOB_B_A_14-22549AB/40X/
   File: SOB_B_A-14-22549AB-40-001.png

COMPRESSION VALIDATION TEST

Image size: 700x460x3 = 966,000 bytes (943.4 KB)
This should be ~700x460 for original BreakHis

CR         Achieved     Size(bytes)  BPP        PSNR(dB)     SSIM       Status
--------------------------------------------------------------------------------
CR15       15.0         64338        1.5985     41.00        0.9744     ✓ GOOD
CR25       25.0         38598        0.9590     38.20        0.9558     OK
CR50       50.0         19313        0.4798     34.84        0.9166     OK
CR100      100.2        9641         0.2395     31.92        0.8405     OK
CR500      510.3        1893         0.0470     27.10        0.6229     ⚠️ POOR
CR1K       1025.5       942          0.0234     25.88        0.5494     ⚠️ POOR
CR5K       3959.0       244          0.0061     11.64        0.4637     ⚠️ UNUSABLE
CR10K      3959.0       244          0.0061     11.64      

In [ ]:
# CELL 5: FULL COMPRESSION PIPELINE FOR ORIGINAL BREAKHIS
import numpy as np
import cv2
import glymur
import tempfile
import os
import pandas as pd
from tqdm import tqdm
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim
import shutil

# Configuration
DATA_ROOT = '/content/drive/MyDrive/BreakHis Original'
OUTPUT_DIR = '/content/drive/MyDrive/BreakHis_Compressed'
COMPRESSION_RATIOS = [15, 25, 50, 100, 500, 1000]  # Excluding 5K+ (hit floor)

CR_NAMES = {15: 'CR15', 25: 'CR25', 50: 'CR50', 100: 'CR100', 500: 'CR500', 1000: 'CR1K'}

# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)

def get_all_images(data_root):
    """Get all images from original BreakHis structure."""
    images = []

    for class_name in ['benign', 'malignant']:
        sob_dir = os.path.join(data_root, class_name, 'SOB')
        if not os.path.exists(sob_dir):
            continue

        for subtype in os.listdir(sob_dir):
            subtype_dir = os.path.join(sob_dir, subtype)
            if not os.path.isdir(subtype_dir):
                continue

            for patient in os.listdir(subtype_dir):
                patient_dir = os.path.join(subtype_dir, patient)
                if not os.path.isdir(patient_dir):
                    continue

                for mag in os.listdir(patient_dir):
                    mag_dir = os.path.join(patient_dir, mag)
                    if not os.path.isdir(mag_dir):
                        continue

                    for filename in os.listdir(mag_dir):
                        if filename.lower().endswith('.png'):
                            # Determine binary label
                            label = 0 if class_name == 'benign' else 1

                            images.append({
                                'filepath': os.path.join(mag_dir, filename),
                                'filename': filename,
                                'class': class_name,
                                'subtype': subtype,
                                'patient_id': patient,
                                'magnification': mag,
                                'label': label
                            })

    return images

# Get all images
print("Scanning dataset...")
all_images = get_all_images(DATA_ROOT)
print(f"Found {len(all_images)} images")

# Print summary
print("\nDataset Summary:")
for cls in ['benign', 'malignant']:
    cls_imgs = [img for img in all_images if img['class'] == cls]
    print(f"  {cls}: {len(cls_imgs)} images")
    subtypes = set(img['subtype'] for img in cls_imgs)
    for st in subtypes:
        st_imgs = [img for img in cls_imgs if img['subtype'] == st]
        print(f"    {st}: {len(st_imgs)}")

# Compression function
def compress_and_save(img_info, compression_ratio, output_dir):
    """Compress image and save, return metrics."""
    img = cv2.imread(img_info['filepath'])
    if img is None:
        return None

    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w, c = img_rgb.shape
    original_bytes = h * w * c

    temp_path = tempfile.mktemp(suffix='.jp2')

    try:
        # Compress
        glymur.Jp2k(temp_path, data=img_rgb, cratios=[compression_ratio])
        compressed_size = os.path.getsize(temp_path)

        # Decompress
        jp2 = glymur.Jp2k(temp_path)
        decompressed = jp2[:]
        if decompressed.dtype != np.uint8:
            decompressed = np.clip(decompressed, 0, 255).astype(np.uint8)

        # Compute metrics
        achieved_cr = original_bytes / compressed_size
        bpp = (compressed_size * 8) / (h * w)
        psnr_val = psnr(img_rgb, decompressed, data_range=255)
        ssim_val = ssim(img_rgb, decompressed, channel_axis=2, data_range=255)

        # Save decompressed image
        cr_name = CR_NAMES.get(compression_ratio, f'CR{compression_ratio}')
        out_subdir = os.path.join(output_dir, cr_name, img_info['class'],
                                   img_info['subtype'], img_info['magnification'])
        os.makedirs(out_subdir, exist_ok=True)

        out_path = os.path.join(out_subdir, img_info['filename'])
        cv2.imwrite(out_path, cv2.cvtColor(decompressed, cv2.COLOR_RGB2BGR))

        return {
            'filename': img_info['filename'],
            'class': img_info['class'],
            'subtype': img_info['subtype'],
            'patient_id': img_info['patient_id'],
            'magnification': img_info['magnification'],
            'label': img_info['label'],
            'compression_ratio': compression_ratio,
            'achieved_cr': achieved_cr,
            'compressed_bytes': compressed_size,
            'bpp': bpp,
            'psnr_db': psnr_val,
            'ssim': ssim_val
        }

    except Exception as e:
        print(f"Error compressing {img_info['filename']}: {e}")
        return None
    finally:
        if os.path.exists(temp_path):
            os.remove(temp_path)

# Copy originals first
print("\nCopying original images...")
original_dir = os.path.join(OUTPUT_DIR, 'CR1_original')
for img_info in tqdm(all_images, desc="Copying originals"):
    out_subdir = os.path.join(original_dir, img_info['class'],
                               img_info['subtype'], img_info['magnification'])
    os.makedirs(out_subdir, exist_ok=True)
    shutil.copy2(img_info['filepath'], os.path.join(out_subdir, img_info['filename']))

# Compress at each ratio
all_metrics = []

for cr in COMPRESSION_RATIOS:
    cr_name = CR_NAMES.get(cr, f'CR{cr}')
    print(f"\nCompressing at {cr_name}...")

    cr_metrics = []
    for img_info in tqdm(all_images, desc=cr_name):
        result = compress_and_save(img_info, cr, OUTPUT_DIR)
        if result:
            cr_metrics.append(result)

    all_metrics.extend(cr_metrics)

    # Print summary for this CR
    if cr_metrics:
        avg_psnr = np.mean([m['psnr_db'] for m in cr_metrics])
        avg_ssim = np.mean([m['ssim'] for m in cr_metrics])
        print(f"  ✓ {len(cr_metrics)} images | Avg PSNR: {avg_psnr:.2f} dB | Avg SSIM: {avg_ssim:.4f}")

# Save metrics
metrics_df = pd.DataFrame(all_metrics)
metrics_path = os.path.join(OUTPUT_DIR, 'compression_metrics.csv')
metrics_df.to_csv(metrics_path, index=False)
print(f"\n✅ Metrics saved to: {metrics_path}")

# Generate summary table
print("\n" + "="*80)
print("RATE-DISTORTION SUMMARY")
print("="*80)

summary = metrics_df.groupby('compression_ratio').agg({
    'achieved_cr': ['mean', 'std'],
    'bpp': ['mean', 'std'],
    'psnr_db': ['mean', 'std'],
    'ssim': ['mean', 'std']
}).round(4)

print(f"\n{'CR':<8} {'Achieved CR':<18} {'BPP':<12} {'PSNR (dB)':<20} {'SSIM'}")
print("-"*80)

for cr in COMPRESSION_RATIOS:
    cr_data = metrics_df[metrics_df['compression_ratio'] == cr]
    cr_name = CR_NAMES.get(cr, f'CR{cr}')
    print(f"{cr_name:<8} "
          f"{cr_data['achieved_cr'].mean():.1f} ± {cr_data['achieved_cr'].std():.1f}      "
          f"{cr_data['bpp'].mean():.4f}      "
          f"{cr_data['psnr_db'].mean():.2f} ± {cr_data['psnr_db'].std():.2f}      "
          f"{cr_data['ssim'].mean():.4f} ± {cr_data['ssim'].std():.4f}")

# Save summary
summary_path = os.path.join(OUTPUT_DIR, 'rate_distortion_summary.csv')
summary.to_csv(summary_path)
print(f"\n✅ Summary saved to: {summary_path}")

print("\n" + "="*80)
print("COMPRESSION COMPLETE!")
print("="*80)
print(f"Output directory: {OUTPUT_DIR}")
print(f"Total images compressed: {len(all_metrics)}")

Scanning dataset...
Found 7910 images

Dataset Summary:
  benign: 2480 images
    phyllodes_tumor: 453
    tubular_adenoma: 569
    fibroadenoma: 1014
    adenosis: 444
  malignant: 5430 images
    mucinous_carcinoma: 793
    ductal_carcinoma: 3451
    papillary_carcinoma: 560
    lobular_carcinoma: 626

Copying original images...


Copying originals: 100%|██████████| 7910/7910 [34:28<00:00,  3.82it/s]



Compressing at CR15...


CR15: 100%|██████████| 7910/7910 [1:24:23<00:00,  1.56it/s]


  ✓ 7910 images | Avg PSNR: 39.32 dB | Avg SSIM: 0.9700

Compressing at CR25...


...

  ✓ 7910 images | Avg PSNR: 30.77 dB | Avg SSIM: 0.8382

Compressing at CR500...


CR500: 100%|██████████| 7910/7910 [50:02<00:00,  2.63it/s]


  ✓ 7910 images | Avg PSNR: 25.02 dB | Avg SSIM: 0.5900

Compressing at CR1K...


CR1K: 100%|██████████| 7910/7910 [49:47<00:00,  2.65it/s]


  ✓ 7910 images | Avg PSNR: 23.20 dB | Avg SSIM: 0.5000

✅ Metrics saved to: /content/drive/MyDrive/BreakHis_Compressed/compression_metrics.csv

RATE-DISTORTION SUMMARY

CR       Achieved CR        BPP          PSNR (dB)            SSIM
--------------------------------------------------------------------------------
CR15     15.0 ± 0.0      1.5971      39.32 ± 3.10      0.9700 ± 0.0097
CR25     25.1 ± 0.1      0.9577      36.83 ± 3.53      0.9508 ± 0.0181
CR50     50.2 ± 0.2      0.4782      33.81 ± 3.81      0.9112 ± 0.0356
CR100    100.7 ± 0.9      0.2384      30.77 ± 3.89      0.8382 ± 0.0657
CR500    507.6 ± 12.5      0.0473      25.02 ± 3.33      0.5900 ± 0.1384
CR1K     1012.8 ± 26.3      0.0237      23.20 ± 3.03      0.5000 ± 0.1552

✅ Summary saved to: /content/drive/MyDrive/BreakHis_Compressed/rate_distortion_summary.csv

COMPRESSION COMPLETE!
Output directory: /content/drive/MyDrive/BreakHis_Compressed
Total images compressed: 47460


In [ ]:
# CELL 6 FIXED: PATIENT-WISE SPLITTING WITH CORRECT ID EXTRACTION
import os
import re
import numpy as np
import pandas as pd
from collections import defaultdict
from sklearn.model_selection import train_test_split
import shutil
from tqdm import tqdm

# Configuration
COMPRESSED_DIR = '/content/drive/MyDrive/BreakHis_Compressed'
OUTPUT_DIR = '/content/drive/MyDrive/BreakHis_Split'
SEED = 42

# Compression levels (including original)
COMPRESSION_LEVELS = ['CR1_original', 'CR15', 'CR25', 'CR50', 'CR100', 'CR500', 'CR1K']

# Split ratios
TRAIN_RATIO = 0.7
VAL_RATIO = 0.15
TEST_RATIO = 0.15

np.random.seed(SEED)

def extract_patient_id(filename):
    """
    Extract patient/slide ID from BreakHis filename.

    Format: <BIOPSY>_<CLASS>_<TYPE>-<YEAR>-<SLIDE_ID>-<MAG>-<SEQ>.png

    Patient ID = YEAR-SLIDE_ID (unique per patient/slide)

    Examples:
        SOB_B_A-14-22549AB-40-001.png    -> 14-22549AB
        SOB_M_DC-14-10926-100-010.png    -> 14-10926
        SOB_B_F-14-9133-200-003.png      -> 14-9133
        SOB_M_MC-14-13418DE-400-001.png  -> 14-13418DE
    """
    # Remove .png extension
    name = filename.replace('.png', '').replace('.PNG', '')

    # Split by '-'
    parts = name.split('-')

    # Format: SOB_B_A-14-22549AB-40-001
    # After split: ['SOB_B_A', '14', '22549AB', '40', '001']
    # Patient ID is parts[1] + '-' + parts[2]

    if len(parts) >= 4:
        year = parts[1]
        slide_id = parts[2]
        patient_id = f"{year}-{slide_id}"
        return patient_id

    print(f"Warning: Could not extract patient ID from {filename}")
    return None


def get_all_images_from_compressed(compressed_dir, cr_level):
    """Get all images from a specific compression level."""
    images = []
    cr_dir = os.path.join(compressed_dir, cr_level)

    if not os.path.exists(cr_dir):
        print(f"Warning: {cr_dir} does not exist")
        return images

    for class_name in ['benign', 'malignant']:
        class_dir = os.path.join(cr_dir, class_name)
        if not os.path.exists(class_dir):
            continue

        for subtype in os.listdir(class_dir):
            subtype_dir = os.path.join(class_dir, subtype)
            if not os.path.isdir(subtype_dir):
                continue

            for mag in os.listdir(subtype_dir):
                mag_dir = os.path.join(subtype_dir, mag)
                if not os.path.isdir(mag_dir):
                    continue

                for filename in os.listdir(mag_dir):
                    if filename.lower().endswith('.png'):
                        patient_id = extract_patient_id(filename)
                        label = 0 if class_name == 'benign' else 1

                        images.append({
                            'filepath': os.path.join(mag_dir, filename),
                            'filename': filename,
                            'class': class_name,
                            'subtype': subtype,
                            'magnification': mag,
                            'patient_id': patient_id,
                            'label': label
                        })

    return images


def create_patient_wise_split(images_df):
    """
    Create patient-wise train/val/test split.
    Ensures no patient appears in multiple splits.
    Stratified by class (benign/malignant).
    """
    # Get unique patients with their class (for stratification)
    patient_info = images_df.groupby('patient_id').agg({
        'label': 'first',  # All images from same patient have same label
        'filename': 'count',  # Number of images per patient
        'class': 'first',
        'subtype': 'first'
    }).reset_index()
    patient_info.columns = ['patient_id', 'label', 'n_images', 'class', 'subtype']

    print(f"\n{'='*60}")
    print("PATIENT STATISTICS")
    print(f"{'='*60}")
    print(f"Total patients: {len(patient_info)}")
    print(f"  Benign patients: {len(patient_info[patient_info['label'] == 0])}")
    print(f"  Malignant patients: {len(patient_info[patient_info['label'] == 1])}")

    print(f"\nPatients per subtype:")
    for subtype in patient_info['subtype'].unique():
        count = len(patient_info[patient_info['subtype'] == subtype])
        print(f"  {subtype}: {count}")

    print(f"\nImages per patient (min/max/mean):")
    print(f"  Min: {patient_info['n_images'].min()}")
    print(f"  Max: {patient_info['n_images'].max()}")
    print(f"  Mean: {patient_info['n_images'].mean():.1f}")

    # First split: train vs (val + test)
    train_patients, temp_patients = train_test_split(
        patient_info['patient_id'].values,
        test_size=(VAL_RATIO + TEST_RATIO),
        random_state=SEED,
        stratify=patient_info['label'].values
    )

    # Second split: val vs test
    temp_info = patient_info[patient_info['patient_id'].isin(temp_patients)]
    val_patients, test_patients = train_test_split(
        temp_info['patient_id'].values,
        test_size=TEST_RATIO / (VAL_RATIO + TEST_RATIO),
        random_state=SEED,
        stratify=temp_info['label'].values
    )

    # Convert to sets for fast lookup
    train_set = set(train_patients)
    val_set = set(val_patients)
    test_set = set(test_patients)

    # Verify no overlap
    assert len(train_set & val_set) == 0, "Train/Val patient overlap detected!"
    assert len(train_set & test_set) == 0, "Train/Test patient overlap detected!"
    assert len(val_set & test_set) == 0, "Val/Test patient overlap detected!"
    print("\n✅ Verified: No patient overlap between splits!")

    print(f"\n{'='*60}")
    print("SPLIT RESULTS")
    print(f"{'='*60}")
    print(f"{'Split':<10} {'Patients':<12} {'Benign':<10} {'Malignant':<10}")
    print("-"*42)

    for split_name, split_patients in [('Train', train_patients), ('Val', val_patients), ('Test', test_patients)]:
        split_info = patient_info[patient_info['patient_id'].isin(split_patients)]
        n_benign = len(split_info[split_info['label'] == 0])
        n_malignant = len(split_info[split_info['label'] == 1])
        print(f"{split_name:<10} {len(split_patients):<12} {n_benign:<10} {n_malignant:<10}")

    # Assign split to each image
    def get_split(patient_id):
        if patient_id in train_set:
            return 'train'
        elif patient_id in val_set:
            return 'val'
        elif patient_id in test_set:
            return 'test'
        else:
            return 'unknown'

    images_df['split'] = images_df['patient_id'].apply(get_split)

    # Print image counts per split
    print(f"\n{'='*60}")
    print("IMAGE COUNTS PER SPLIT")
    print(f"{'='*60}")
    print(f"{'Split':<10} {'Images':<12} {'Benign':<10} {'Malignant':<10}")
    print("-"*42)

    for split in ['train', 'val', 'test']:
        split_df = images_df[images_df['split'] == split]
        n_benign = len(split_df[split_df['label'] == 0])
        n_malignant = len(split_df[split_df['label'] == 1])
        print(f"{split.capitalize():<10} {len(split_df):<12} {n_benign:<10} {n_malignant:<10}")

    return images_df, list(train_patients), list(val_patients), list(test_patients)


def copy_images_to_split_structure(images_df, cr_level, output_dir):
    """Copy images to train/val/test folder structure."""
    cr_output = os.path.join(output_dir, cr_level)

    for _, row in tqdm(images_df.iterrows(), total=len(images_df), desc=f"Copying {cr_level}", leave=False):
        # Create output path: CR_level/split/class/magnification/filename
        out_subdir = os.path.join(cr_output, row['split'], row['class'], row['magnification'])
        os.makedirs(out_subdir, exist_ok=True)

        out_path = os.path.join(out_subdir, row['filename'])
        shutil.copy2(row['filepath'], out_path)


# ============================================
# MAIN EXECUTION
# ============================================

print("="*70)
print("PATIENT-WISE DATASET SPLITTING")
print("="*70)

# Step 1: Test patient ID extraction
print("\nStep 1: Testing patient ID extraction...")
test_filenames = [
    'SOB_B_A-14-22549AB-40-001.png',
    'SOB_M_DC-14-10926-100-010.png',
    'SOB_B_F-14-9133-200-003.png',
    'SOB_M_MC-14-13418DE-400-001.png',
    'SOB_B_PT-14-21998AB-200-016.png'
]

print(f"{'Filename':<45} {'Extracted Patient ID'}")
print("-"*65)
for fn in test_filenames:
    pid = extract_patient_id(fn)
    print(f"{fn:<45} {pid}")

# Step 2: Get images from original (CR1) to determine patient split
print(f"\n{'='*70}")
print("Step 2: Loading images from CR1_original...")
print("="*70)

original_images = get_all_images_from_compressed(COMPRESSED_DIR, 'CR1_original')
original_df = pd.DataFrame(original_images)

print(f"Found {len(original_df)} images")
print(f"Unique patients: {original_df['patient_id'].nunique()}")

# Check for any failed patient ID extractions
null_patients = original_df[original_df['patient_id'].isnull()]
if len(null_patients) > 0:
    print(f"\n⚠️ Warning: {len(null_patients)} images have no patient ID")
    print("Sample filenames:")
    for fn in null_patients['filename'].head(5).values:
        print(f"  {fn}")
else:
    print("✅ All images have valid patient IDs")

# Remove images without patient ID
original_df = original_df[original_df['patient_id'].notnull()]

# Step 3: Create patient-wise split
print(f"\n{'='*70}")
print("Step 3: Creating patient-wise split...")
print("="*70)

original_df, train_patients, val_patients, test_patients = create_patient_wise_split(original_df)

# Save split information
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Save patient lists with details
patient_split_df = pd.DataFrame({
    'patient_id': train_patients + val_patients + test_patients,
    'split': ['train'] * len(train_patients) + ['val'] * len(val_patients) + ['test'] * len(test_patients)
})

# Add class info
patient_class = original_df.groupby('patient_id')['class'].first().to_dict()
patient_split_df['class'] = patient_split_df['patient_id'].map(patient_class)

patient_split_df.to_csv(os.path.join(OUTPUT_DIR, 'patient_splits.csv'), index=False)
print(f"\n✅ Patient split saved to: {OUTPUT_DIR}/patient_splits.csv")

# Step 4: Apply SAME split to all compression levels
print(f"\n{'='*70}")
print("Step 4: Applying split to all compression levels...")
print("="*70)

all_split_stats = []

for cr_level in COMPRESSION_LEVELS:
    print(f"\nProcessing {cr_level}...")

    # Get images for this CR level
    cr_images = get_all_images_from_compressed(COMPRESSED_DIR, cr_level)
    cr_df = pd.DataFrame(cr_images)

    if len(cr_df) == 0:
        print(f"  ⚠️ No images found for {cr_level}, skipping...")
        continue

    # Remove images without patient ID
    cr_df = cr_df[cr_df['patient_id'].notnull()]

    # Apply the SAME patient split
    train_set = set(train_patients)
    val_set = set(val_patients)
    test_set = set(test_patients)

    def get_split(patient_id):
        if patient_id in train_set:
            return 'train'
        elif patient_id in val_set:
            return 'val'
        elif patient_id in test_set:
            return 'test'
        else:
            return 'unknown'

    cr_df['split'] = cr_df['patient_id'].apply(get_split)

    # Check for unknown splits
    unknown = cr_df[cr_df['split'] == 'unknown']
    if len(unknown) > 0:
        print(f"  ⚠️ {len(unknown)} images have unknown split")

    cr_df = cr_df[cr_df['split'] != 'unknown']

    # Copy images to split structure
    copy_images_to_split_structure(cr_df, cr_level, OUTPUT_DIR)

    # Collect stats
    for split in ['train', 'val', 'test']:
        split_data = cr_df[cr_df['split'] == split]
        all_split_stats.append({
            'cr_level': cr_level,
            'split': split,
            'n_images': len(split_data),
            'n_benign': len(split_data[split_data['label'] == 0]),
            'n_malignant': len(split_data[split_data['label'] == 1]),
            'n_patients': split_data['patient_id'].nunique()
        })

    print(f"  ✓ {len(cr_df)} images copied")

# Save split statistics
stats_df = pd.DataFrame(all_split_stats)
stats_df.to_csv(os.path.join(OUTPUT_DIR, 'split_statistics.csv'), index=False)

# Print final summary
print(f"\n{'='*70}")
print("FINAL SUMMARY")
print("="*70)

# Pivot table for nice display
pivot = stats_df.pivot_table(
    index='cr_level',
    columns='split',
    values='n_images',
    aggfunc='sum'
)[['train', 'val', 'test']]

print("\nImages per CR level and split:")
print(pivot.to_string())

print(f"\n✅ Output directory: {OUTPUT_DIR}")
print(f"✅ Split statistics saved to: {OUTPUT_DIR}/split_statistics.csv")
print(f"✅ Patient splits saved to: {OUTPUT_DIR}/patient_splits.csv")

print(f"\n{'='*70}")
print("OUTPUT FOLDER STRUCTURE")
print("="*70)
print("""
BreakHis_Split/
├── CR1_original/
│   ├── train/
│   │   ├── benign/
│   │   │   ├── 40X/
│   │   │   ├── 100X/
│   │   │   ├── 200X/
│   │   │   └── 400X/
│   │   └── malignant/
│   │       └── ...
│   ├── val/
│   └── test/
├── CR15/
│   ├── train/
│   ├── val/
│   └── test/
├── CR25/
├── CR50/
├── CR100/
├── CR500/
├── CR1K/
├── patient_splits.csv      # Patient ID -> split mapping
└── split_statistics.csv    # Image counts per split
""")

print("\n" + "="*70)
print("⚠️  CRITICAL: SAME patients in train/val/test across ALL compression levels")
print("    This ensures FAIR and VALID comparison between compression ratios!")
print("="*70)

PATIENT-WISE DATASET SPLITTING

Step 1: Testing patient ID extraction...
Filename                                      Extracted Patient ID
-----------------------------------------------------------------
SOB_B_A-14-22549AB-40-001.png                 14-22549AB
SOB_M_DC-14-10926-100-010.png                 14-10926
SOB_B_F-14-9133-200-003.png                   14-9133
SOB_M_MC-14-13418DE-400-001.png               14-13418DE
SOB_B_PT-14-21998AB-200-016.png               14-21998AB

Step 2: Loading images from CR1_original...
Found 7910 images
Unique patients: 81
✅ All images have valid patient IDs

Step 3: Creating patient-wise split...

PATIENT STATISTICS
Total patients: 81
  Benign patients: 24
  Malignant patients: 57

Patients per subtype:
  mucinous_carcinoma: 9
  ductal_carcinoma: 38
  lobular_carcinoma: 4
  papillary_carcinoma: 6
  tubular_adenoma: 7
  fibroadenoma: 10
  phyllodes_tumor: 3
  adenosis: 4

Images per patient (min/max/mean):
  Min: 38
  Max: 246
  Mean: 97.7

✅ Ver

  ✓ 7910 images copied

Processing CR15...


  ✓ 7910 images copied

Processing CR25...


  ✓ 7910 images copied

Processing CR50...


  ✓ 7910 images copied

Processing CR100...


  ✓ 7910 images copied

Processing CR500...


  ✓ 7910 images copied

Processing CR1K...


Copying CR1K:   2%|▏         | 152/7910 [01:08<1:27:49,  1.47it/s]

In [3]:
# RECOVERY SCRIPT: Complete CR1K splitting only
import os
import re
import numpy as np
import pandas as pd
import shutil
from tqdm import tqdm

# Configuration
COMPRESSED_DIR = '/content/drive/MyDrive/BreakHis_Compressed'
OUTPUT_DIR = '/content/drive/MyDrive/BreakHis_Split'
SEED = 42

np.random.seed(SEED)

def extract_patient_id(filename):
    """Extract patient ID from BreakHis filename."""
    name = filename.replace('.png', '').replace('.PNG', '')
    parts = name.split('-')
    if len(parts) >= 4:
        return f"{parts[1]}-{parts[2]}"
    return None

def get_all_images_from_compressed(compressed_dir, cr_level):
    """Get all images from a specific compression level."""
    images = []
    cr_dir = os.path.join(compressed_dir, cr_level)

    if not os.path.exists(cr_dir):
        return images

    for class_name in ['benign', 'malignant']:
        class_dir = os.path.join(cr_dir, class_name)
        if not os.path.exists(class_dir):
            continue

        for subtype in os.listdir(class_dir):
            subtype_dir = os.path.join(class_dir, subtype)
            if not os.path.isdir(subtype_dir):
                continue

            for mag in os.listdir(subtype_dir):
                mag_dir = os.path.join(subtype_dir, mag)
                if not os.path.isdir(mag_dir):
                    continue

                for filename in os.listdir(mag_dir):
                    if filename.lower().endswith('.png'):
                        patient_id = extract_patient_id(filename)
                        label = 0 if class_name == 'benign' else 1

                        images.append({
                            'filepath': os.path.join(mag_dir, filename),
                            'filename': filename,
                            'class': class_name,
                            'subtype': subtype,
                            'magnification': mag,
                            'patient_id': patient_id,
                            'label': label
                        })

    return images

# Load existing patient splits
print("Loading existing patient splits...")
patient_splits_df = pd.read_csv(os.path.join(OUTPUT_DIR, 'patient_splits.csv'))

train_patients = patient_splits_df[patient_splits_df['split'] == 'train']['patient_id'].tolist()
val_patients = patient_splits_df[patient_splits_df['split'] == 'val']['patient_id'].tolist()
test_patients = patient_splits_df[patient_splits_df['split'] == 'test']['patient_id'].tolist()

train_set = set(train_patients)
val_set = set(val_patients)
test_set = set(test_patients)

print(f"✅ Loaded splits: {len(train_patients)} train, {len(val_patients)} val, {len(test_patients)} test patients")

# Process only CR1K
print("\n" + "="*60)
print("Completing CR1K splitting...")
print("="*60)

cr_level = 'CR1K'
cr_images = get_all_images_from_compressed(COMPRESSED_DIR, cr_level)
cr_df = pd.DataFrame(cr_images)
cr_df = cr_df[cr_df['patient_id'].notnull()]

def get_split(patient_id):
    if patient_id in train_set:
        return 'train'
    elif patient_id in val_set:
        return 'val'
    elif patient_id in test_set:
        return 'test'
    else:
        return 'unknown'

cr_df['split'] = cr_df['patient_id'].apply(get_split)
cr_df = cr_df[cr_df['split'] != 'unknown']

# Copy only images that don't already exist
cr_output = os.path.join(OUTPUT_DIR, cr_level)
copied = 0
skipped = 0

for _, row in tqdm(cr_df.iterrows(), total=len(cr_df), desc=f"Copying {cr_level}"):
    out_subdir = os.path.join(cr_output, row['split'], row['class'], row['magnification'])
    os.makedirs(out_subdir, exist_ok=True)

    out_path = os.path.join(out_subdir, row['filename'])

    # Skip if already exists
    if os.path.exists(out_path):
        skipped += 1
        continue

    shutil.copy2(row['filepath'], out_path)
    copied += 1

print(f"\n✅ CR1K complete: {copied} copied, {skipped} already existed")

# Generate final statistics
print("\n" + "="*60)
print("Generating final statistics...")
print("="*60)

COMPRESSION_LEVELS = ['CR1_original', 'CR15', 'CR25', 'CR50', 'CR100', 'CR500', 'CR1K']
all_split_stats = []

for cr_level in COMPRESSION_LEVELS:
    cr_output = os.path.join(OUTPUT_DIR, cr_level)

    for split in ['train', 'val', 'test']:
        split_dir = os.path.join(cr_output, split)

        n_images = 0
        n_benign = 0
        n_malignant = 0

        if os.path.exists(split_dir):
            for class_name in ['benign', 'malignant']:
                class_dir = os.path.join(split_dir, class_name)
                if os.path.exists(class_dir):
                    for mag in os.listdir(class_dir):
                        mag_dir = os.path.join(class_dir, mag)
                        if os.path.isdir(mag_dir):
                            count = len([f for f in os.listdir(mag_dir) if f.endswith('.png')])
                            n_images += count
                            if class_name == 'benign':
                                n_benign += count
                            else:
                                n_malignant += count

        all_split_stats.append({
            'cr_level': cr_level,
            'split': split,
            'n_images': n_images,
            'n_benign': n_benign,
            'n_malignant': n_malignant
        })

stats_df = pd.DataFrame(all_split_stats)
stats_df.to_csv(os.path.join(OUTPUT_DIR, 'split_statistics.csv'), index=False)

# Print final summary
print("\n" + "="*70)
print("FINAL SUMMARY")
print("="*70)

print(f"\n{'CR Level':<15} {'Split':<8} {'Images':<10} {'Benign':<10} {'Malignant':<10}")
print("-"*55)

for cr_level in COMPRESSION_LEVELS:
    cr_stats = stats_df[stats_df['cr_level'] == cr_level]
    for _, row in cr_stats.iterrows():
        print(f"{row['cr_level']:<15} {row['split']:<8} {row['n_images']:<10} {row['n_benign']:<10} {row['n_malignant']:<10}")
    print("-"*55)

# Verify totals
print("\n" + "="*70)
print("VERIFICATION")
print("="*70)

for cr_level in COMPRESSION_LEVELS:
    cr_stats = stats_df[stats_df['cr_level'] == cr_level]
    total = cr_stats['n_images'].sum()
    expected = 7910
    status = "✅" if total == expected else "❌"
    print(f"{cr_level}: {total}/{expected} images {status}")

print(f"\n✅ Split statistics saved to: {OUTPUT_DIR}/split_statistics.csv")
print(f"✅ Patient splits saved to: {OUTPUT_DIR}/patient_splits.csv")

print("\n" + "="*70)
print("SPLITTING COMPLETE!")
print("="*70)

Loading existing patient splits...
✅ Loaded splits: 56 train, 12 val, 13 test patients

Completing CR1K splitting...


Copying CR1K: 100%|██████████| 7910/7910 [1:06:30<00:00,  1.98it/s]



✅ CR1K complete: 6285 copied, 1625 already existed

Generating final statistics...

FINAL SUMMARY

CR Level        Split    Images     Benign     Malignant 
-------------------------------------------------------
CR1_original    train    5491       1818       3673      
CR1_original    val      1055       197        858       
CR1_original    test     1364       465        899       
-------------------------------------------------------
CR15            train    5491       1818       3673      
CR15            val      1055       197        858       
CR15            test     1364       465        899       
-------------------------------------------------------
CR25            train    5491       1818       3673      
CR25            val      1055       197        858       
CR25            test     1364       465        899       
-------------------------------------------------------
CR50            train    5491       1818       3673      
CR50            val      1055       19